# 8 (Capstone) - From Design to Training Data

Chapter 8 puts a composed design to its first use: generating a supervised corpus. The capstone trains two of its tools this way --- `classify_complaint` and `draft_response` are fine-tuned on DoE-generated data --- so the two emitters of this chapter are exactly how that data was made.

This notebook takes the 120-scenario suite of Chapter 7 and runs both emitters on it: a **label-preserving classifier corpus**, where enrichment varies the surface and the label is fixed by construction, and **grounded draft pairs** validated against a golden contract that drops any response that is not faithful to policy.

## Consumer A: a label-preserving classifier corpus

We rebuild the Chapter 7 suite, then emit classifier records. The message is *materialized* from the design row by the capstone's own factor transforms (clarity, aliasing, reasoning cue); the label is read from the case's answer key. Because every factor is ground-truth invariant, the label is fixed no matter how the surface is varied --- a classifier corpus synthesized without any re-labeling.

**What the next cell does** --- rebuild the Chapter 7 suite and emit the label-preserving classifier corpus:

1. **Locate the roots.** Walk up from the cwd to find `TUT` (the tutorial dir holding `apps/complaint_agent.yaml`) and `ROOT` (the repo holding `data/eval_cases/cases.json`), then put both on `sys.path` and import `gmstest as gt` plus `CapstoneTestHarness as H`.
2. **Resolve and compose the suite.** Load the catalog with `gt.Catalog.load()`, read the 20 seed cases with `gt.SeedCaseSource(...).items()`, pull the `parity_ch16` factor block from the YAML, `gt.resolve` the `conditional_rule` category against those factors, then `gt.compose` 120 scenarios with `graphdoe_design(method='sobol+refine')` (balanced base, `level_overrides` from the YAML, seed 42).
3. **Materialize each row into a message.** `materialize(s)` chains the capstone's own transforms `H._apply_clarity`, `H._apply_aliasing` and `H._apply_cue` over `s.base.query`, driven by `s.factor_levels`, so the surface is varied without a Qwen rephraser.
4. **Emit and inspect.** `gt.emit_classifier_sft(scns, materialize_fn=materialize)` produces `{message, label, _seed, _factors}` records (label read from the case answer key), then it prints the record count, the schema, the label histogram and the first three messages.

In [ ]:
import sys, pathlib, json, yaml
from functools import partial
from collections import Counter, defaultdict

TUT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
           if (p / 'code' / 'apps' / 'complaint_agent.yaml').exists()) / 'code'
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'beyond-prompt-and-pray' / 'code' / 'data' / 'eval_cases' / 'cases.json').exists()) / 'beyond-prompt-and-pray' / 'code'
sys.path.insert(0, str(TUT))
sys.path.insert(0, str(ROOT))
import gmstest as gt
from gmstest.compose import graphdoe_design
from agentlab.testing.capstone_harness import CapstoneTestHarness as H

cat = gt.Catalog.load()
items = gt.SeedCaseSource(json.loads((ROOT / 'data' / 'eval_cases' / 'cases.json').read_text())).items()
pc = yaml.safe_load((TUT / 'apps' / 'complaint_agent.yaml').read_text())['parity_ch16']
suite = gt.resolve(cat, ['conditional_rule'], pc['factors'], mode='embedded')
scns = gt.compose(suite, items, n_runs=120, seed=42,
                  design_fn=partial(graphdoe_design, method='sobol+refine'),
                  balance_base=True, level_overrides=pc['level_overrides'])

# materialize the design row into a message with the capstone's own transforms
def materialize(s):
    m = H._apply_clarity(s.base.query, s.factor_levels['clarity'])
    m = H._apply_aliasing(m, s.factor_levels['entity_aliasing'])
    return H._apply_cue(m, s.factor_levels['reasoning_cue'])

recs = gt.emit_classifier_sft(scns, materialize_fn=materialize)
print(f'{len(recs)} classifier records | schema {sorted(recs[0])}')
print('labels present :', dict(Counter(r['label'] for r in recs)), '\n')
for r in recs[:3]:
    print(f"  [{r['label']:9s}] ({r['_factors']['clarity']}/{r['_factors']['entity_aliasing']}) {r['message'][:64]}")

## The label survives every phrasing

The corpus's value is that one case yields many surface variants under a single, fixed label. We group the records by their seed case and confirm each case carries exactly one label across all its phrasings --- which is why no human ever re-labels the augmented data. This is the same `{message, label, _seed, _factors}` schema as the capstone's shipped training file `data/training/complaint_classification/train_doe_augmented.jsonl`.

**What the next cell does** --- confirm the label is invariant across phrasings and cross-check the shipped corpus:

1. **Group by seed case.** Build `by_case`, a `defaultdict(set)` mapping each `r['_seed']` to the set of labels its records carry, then keep in `one_label` the cases whose label set has size one, and print how many of the 20 cases carry exactly one label.
2. **Show one case's variants.** Filter `recs` to `case-001` and print its single label alongside the first four distinct phrasings of that message.
3. **Cross-check the shipped file.** Load `data/training/complaint_classification/train_doe_augmented.jsonl`, then print its record count and `sorted` schema to confirm it uses the same `{message, label, _seed, _factors}` emitter schema.

In [ ]:
by_case = defaultdict(set)
for r in recs:
    by_case[r['_seed']].add(r['label'])
one_label = {q: labs for q, labs in by_case.items() if len(labs) == 1}
print(f'{len(by_case)} cases; each carries exactly one label across its phrasings: '
      f'{len(one_label)}/{len(by_case)}\n')

# the same case, several phrasings, one invariant label
ex = [r for r in recs if r['_seed'] == 'case-001']
print(f"case-001 -> label '{ex[0]['label']}' across {len(ex)} phrasings:")
for r in ex[:4]:
    print('   ', r['message'][:70])

# cross-check the shipped corpus uses this exact emitter schema
shipped = [json.loads(l) for l in
           (ROOT / 'data/training/complaint_classification/train_doe_augmented.jsonl')
           .read_text().splitlines()]
print(f"\nshipped corpus: {len(shipped)} records, schema {sorted(shipped[0])}")

## Consumer B: grounded draft pairs with a golden contract

The draft emitter produces `(user, assistant)` pairs for generative fine-tuning, but it does not trust the generated text: each pair must pass a **golden contract** --- cite the governing policy, hold the byte-exact figure the oracle records, commit nothing prohibited. Pairs that fail are dropped, so the surviving corpus is grounded by construction. We show a faithful response kept and an unauthorized-waiver response dropped, against the capstone's overdraft contract.

**What the next cell does** --- emit grounded draft pairs and show the golden contract keep one and drop one:

1. **Define the contract.** `goldens['overdraft']` sets `citation_keywords=['overdraft']`, `required_numbers=['35']` and `forbidden_phrases=['we will waive']` --- the byte-exact strictness `passes_contract` enforces.
2. **Compose one policy scenario.** Build a single `gt.QAItem` for the overdraft policy and `gt.compose` it (cross design over `clarity`, `n_runs=1`) into `psc`.
3. **Define two responders.** `faithful` cites the overdraft policy and holds `$35`; `unauthorized` promises a waiver.
4. **Emit and compare.** Call `gt.emit_draft_sft(psc, ..., goldens=goldens)` twice; it returns `(kept, dropped)` per responder, validating each `(user, assistant)` pair against the contract. The print shows the faithful pair kept and the unauthorized-waiver pair dropped.

In [ ]:
# the capstone's overdraft golden contract: cite 'overdraft', hold $35, never promise a waiver
goldens = {'overdraft': {'citation_keywords': ['overdraft'],
                         'required_numbers': ['35'],
                         'forbidden_phrases': ['we will waive']}}
pol = [gt.QAItem(qid='p0', query='I was charged a $35 overdraft fee.',
                 answer=None, metadata={'policy_id': 'overdraft', 'issue': 'overdraft_fee'})]
psc = gt.compose(gt.resolve(cat, ['conditional_rule'], ['clarity'], mode='cross'),
                 pol, n_runs=1, seed=42, level_overrides=pc['level_overrides'])

faithful = lambda s: 'Per the overdraft policy, the fee is $35; I can review your account.'
unauthorized = lambda s: 'No problem, we will waive the fee for you right away.'

kept_ok, drop_ok = gt.emit_draft_sft(psc, faithful, goldens=goldens)
kept_bad, drop_bad = gt.emit_draft_sft(psc, unauthorized, goldens=goldens)
print('faithful response     -> kept', len(kept_ok), 'dropped', len(drop_ok))
print('unauthorized waiver   -> kept', len(kept_bad), 'dropped', len(drop_bad),
      '  (golden contract rejects it)')

## Verify the extraction against ground truth, and filter the training pairs

The grounded query `extract_facts` produces is not a classifier label --- Chapter 15 grades it by **downstream success**: did the fact/query *retrieve the right policy* and *reach the right escalation*, not whether a head string matched a taxonomy. So we verify a rephrasing the same way --- against the case's **ground truth** (its acceptable policies) --- on the LLM-free geometric path (parse-bind + retrieve, no Qwen):

1. **Check the original first.** The canonical message must recover a ground-truth policy --- the precondition; if the unrephrased query already misses, the case, not the rephrasing, is at fault.
2. **Test each rephrasing against that same ground truth.** It passes if its grounded query retrieves a policy in the case's acceptable set.
3. **Filter for training data.** A rephrasing that recovers the ground truth becomes a `{query, label}` parse-bind training pair; one that does not is **filtered out** --- it would teach the encoder a wrong binding. (The queries-that-match-the-store rule: matches become pairs, non-matches are dropped.)

knowlytix exposes this as `grounding_preserved` / `filter_grounding_preserving`: with `match="any"` against the acceptable set and a policy-stem `normalize`, a query is kept when it recovers *any* acceptable policy along the GMS path.

**What the next cell does** --- verify each rephrasing against ground truth on the LLM-free geometric path and filter the training pairs:

1. **Load the retriever and the case.** Import `grounding_preserved` and `filter_grounding_preserving` from `knowlytix.knowledge.rag`, get the LLM-free parse-bind retriever with `get_default_retriever()`, pull `case-001`, take its message as `base`, and read its ground-truth set `gt_policies` from `acceptable_policies` (or `expected_policy`). `stem` normalizes a head to its policy stem.
2. **Check the precondition.** `grounding_preserved(retr.pipe, base, expected=gt_policies, match='any', normalize=stem)` confirms the canonical query recovers an acceptable policy along the GMS path; `match='any'` makes "recover any acceptable policy" the pass test.
3. **Test each rephrasing.** For each level in `LEVELS`, `rephrase(*lv)` applies `H._apply_clarity/_aliasing/_cue`, then `grounding_preserved` records `variant_heads` and `preserved` against the same ground truth.
4. **Filter for clean training data.** Build a `corpus` of `{level, message, expected}` and call `filter_grounding_preserving(retr.pipe, corpus, match='any', normalize=stem)`; `res.kept` become parse-bind training pairs and `res.dropped` (failing ground truth) are filtered out, then both lists are printed.

In [ ]:
from knowlytix.knowledge.rag import grounding_preserved, filter_grounding_preserving
from agentlab.capstone.policy_rag import get_default_retriever

retr = get_default_retriever()          # LLM-free geometric parse-bind + retrieve (no Qwen)
case = next(c for c in json.loads((ROOT / 'data' / 'eval_cases' / 'cases.json').read_text())
            if c['id'] == 'case-001')
base = case['message']
gt_policies = set(case.get('acceptable_policies') or [case['expected_policy']])   # ground truth
stem = lambda h: h.split('/')[0]        # compare on the policy stem (overdraft/per_occurrence -> overdraft)

def rephrase(clarity, aliasing, cue):
    m = H._apply_clarity(base, clarity); m = H._apply_aliasing(m, aliasing)
    return H._apply_cue(m, cue)

# 1. precondition: the ORIGINAL query must recover a ground-truth policy.
#    match='any' = downstream success (retrieve ANY acceptable policy), the Ch15 frame.
base_gc = grounding_preserved(retr.pipe, base, expected=gt_policies, match='any', normalize=stem)
print(f"ground truth (acceptable): {sorted(gt_policies)}")
print(f"canonical recovers GT: {base_gc.preserved} -> {base_gc.variant_heads}\n")

# 2. each rephrasing, downstream success vs the SAME ground truth
LEVELS = {'canonical': ('clear', 'canonical', 'none'), 'alias': ('clear', 'alias', 'none'),
          'typo': ('clear', 'typo', 'none'), 'misleading': ('misleading', 'canonical', 'none'),
          'cot': ('clear', 'canonical', 'cot')}
checks = {n: grounding_preserved(retr.pipe, rephrase(*lv), expected=gt_policies,
                                 match='any', normalize=stem) for n, lv in LEVELS.items()}
print(f"{'rephrasing':12s}{'retrieves':>26s}{'GT ok':>8s}")
for n, gc in checks.items():
    print(f"  {n:10s}{str(gc.variant_heads):>26s}{str(gc.preserved):>8s}")

# 3. filter for clean parse-bind training data (the one-call batch form)
corpus = [{'level': n, 'message': rephrase(*lv), 'expected': gt_policies} for n, lv in LEVELS.items()]
res = filter_grounding_preserving(retr.pipe, corpus, match='any', normalize=stem)
kept = [it['level'] for it in res.kept]
dropped = [it['level'] for it, _ in res.dropped]
print(f"\nkept as parse-bind training pairs : {kept}")
print(f"filtered out (fail ground truth)  : {dropped}")

The canonical query recovers `overdraft` --- the precondition holds. Clarity and the reasoning cue keep it there. The `typo` "overdaft" recovers **`fee_reversal`** --- a *drift* by strict head-matching, but `fee_reversal` is in the case's acceptable set and maps to the same `overdraft_fee` issue, so downstream it is a genuine **success** that a strict label-match would have wrongly rejected. Only the out-of-vocabulary abbreviation "OD charge" fails: it grounds to `stop_payment`, outside the acceptable set, so it is **filtered out** of the parse-bind training data rather than teaching the encoder a wrong binding.

This is Chapter 15's frame exactly: the extraction is graded by what it lets the workflow do, not by matching a label --- which is why the entity-aliasing factor never reached significance on the final outcome (Chapter 10). The one genuinely unrecoverable form (an out-of-vocabulary customer abbreviation) is the narrow target for hardening: extend the store's alias vocabulary, not the encoder architecture.

These two corpora are what train the capstone's `classify_complaint` (LoRA + head) and `draft_response` (LoRA) tools in the companion builder volume. The *same* 120 scenarios also feed the chapter's second consumer --- the evaluation run of Chapters 9 and 10, which sends each scenario through the agent and attributes the result. One design, two uses: synthesize the training corpus, and run the test.

## Self-check

The asserts below pin every claim this notebook makes --- 120 classifier records on the shipped `{message, label, _seed, _factors}` schema, every case carrying a single invariant label across its phrasings, the shipped corpus on the same schema, the golden contract keeping a faithful draft while dropping an unauthorized waiver, and the extraction verification: the canonical query recovers ground truth, most rephrasings recover it downstream (the `typo` via an acceptable policy), and the out-of-vocabulary abbreviation is filtered out. Run the notebook end to end and it fails loud if any of these drift.

**What the next cell does** --- assert every claim the notebook makes, so it fails loud on any drift:

1. **Classifier corpus.** Assert `recs` has 120 records on the `{message, label, _seed, _factors}` schema and that every one of the 20 cases carries a single label (`one_label == by_case`).
2. **Shipped corpus.** Assert the shipped JSONL uses the same emitter schema.
3. **Golden contract.** Assert the faithful draft was kept (`kept_ok==1`, `drop_ok==0`) and the unauthorized waiver dropped (`kept_bad==0`, `drop_bad==1`).
4. **Downstream-success verification.** Assert the canonical query recovered ground truth (`base_gc.preserved`), `canonical` and `typo` are in `kept`, the out-of-vocabulary `alias` is in `dropped`, and at least four rephrasings recovered the ground truth; print the OK line.

In [ ]:
assert len(recs) == 120 and set(recs[0]) == {'message', 'label', '_seed', '_factors'}
assert len(one_label) == len(by_case) == 20            # every case: one label across phrasings
assert set(shipped[0]) == {'message', 'label', '_seed', '_factors'}   # shipped corpus, same emitter
assert len(kept_ok) == 1 and len(drop_ok) == 0         # faithful draft kept
assert len(kept_bad) == 0 and len(drop_bad) == 1       # unauthorized waiver dropped by the contract
# downstream-success verification (Chapter 15's frame), against the case's ground truth
assert base_gc.preserved                               # the original query recovers ground truth
assert 'canonical' in kept and 'typo' in kept          # typo->fee_reversal is acceptable -> a success
assert 'alias' in dropped                              # the OOV abbreviation fails -> filtered out
assert len(kept) >= 4                                  # most rephrasings recover the GT downstream
print('OK - Ch8 capstone: corpora + downstream-success verification of the extraction')